In [8]:
import os
from typing import Type

from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.tools import BaseTool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import requests


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

alpha_vantage_api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(description="The query you will search for")


class StockMarketSymbolSearchTool(BaseTool):
    name: str = "StockMarketSymbolSearchTool"
    description: str = """
    Use this tool to find the stock market symbol for a company.
    It takes a query as an argument.
    Example query: Stock Market Symbol for Apple Company
    """

    args_schema: Type[StockMarketSymbolSearchToolArgsSchema] = StockMarketSymbolSearchToolArgsSchema

    def _run(self, query: str):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(query)


class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA, etc.")


class CompanyOverviewTool(BaseTool):
    name: str = "CompanyOverview"
    description: str = """
    Use this tool to get an overview of the financials of the company.
    You should enter a stock symbol.
    """

    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol: str):
        response = requests.get(f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}")

        return response.json()


class CompanyIncomeStatementTool(BaseTool):
    name: str = "CompanyIncomeStatement"
    description: str = """
    Use this tool to get the income statement of the company.
    You should enter a stock symbol.
    """

    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol: str):
        response = requests.get(f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}")

        data = response.json()
        
        if "annualReports" not in data:
          return {
              "error": "No annualReports found in Alpha Vantage response.",
              "response": data,
          }

        return data["annualReports"]


class CompanyStockPerformanceTool(BaseTool):
    name: str = "CompanyStockPerformanceTool"
    description: str = """
    Use this tool to get the weekly performance of a company stock.
    You should enter a stock symbol.
    """

    args_schema: Type[CompanyOverviewToolArgsSchema] = CompanyOverviewToolArgsSchema

    def _run(self, symbol: str):
        response = requests.get(f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_key}")

        return response.json()


class CalculatorToolArgsSchema(BaseModel):
    a: float = Field(description="The first number")
    b: float = Field(description="The second number")


class CalculatorTool(BaseTool):
    name: str = "CalculatorTool"
    description: str = """
    Use this tool to perform sums of two numbers.
    The first and second arguments should be numbers.
    Only receive two arguments.
    """
    args_schema: Type[CalculatorToolArgsSchema] = CalculatorToolArgsSchema

    def _run(self, a: float, b: float) -> float:
        return a + b


agent = create_agent(
    model=llm,
    debug=True,
    # agent=AgentType.OPENAI_FUNCTIONS, # OpenAI Functions 에이전트 유형 사용
    # handle_parsing_errors=True, # 파싱 오류 발생 시 경고 메시지 출력
    tools=[
        StockMarketSymbolSearchTool(),
        CompanyOverviewTool(),
        CompanyIncomeStatementTool(),
        CompanyStockPerformanceTool(),
    ],
)

prompt = "Give me financial information on Cloudflare's stock, considering its financials and income statements and stock performance help me analyze if it's a potential good investment."

result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
display(Markdown(result["messages"][-1].content))


[values] {'messages': [HumanMessage(content="Give me financial information on Cloudflare's stock, considering its financials and income statements and stock performance help me analyze if it's a potential good investment.", additional_kwargs={}, response_metadata={}, id='6739a2c5-270c-4b35-b49d-083e7e9256bb')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 266, 'total_tokens': 284, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ef2d618cea', 'id': 'chatcmpl-DrQ4KPdtp0TnvZo5Mc0wwjW4nDHDH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed110-41bb-7fd1-8f27-fd7da413e800-0', 

It seems that there was an issue retrieving the financial overview and income statement for Cloudflare (symbol: NET). However, I do have the stock performance data for the company. Let's analyze the stock performance data to help assess whether Cloudflare could be a good investment.

### Stock Performance Overview (Recent Weekly Data)

1. **Latest Closing Price**: 
   - **June 15, 2026**: Closed at **$235.71**
   
2. **Weekly Price Movement**:
   - **June 12, 2026**: Closed at **$228.48**
   - **June 5, 2026**: Closed at **$250.11**
   - **May 29, 2026**: Closed at **$241.82**
   - **May 22, 2026**: Closed at **$216.17**
   - **May 15, 2026**: Closed at **$197.56**

3. **Price Trends**:
   - The stock has shown a significant increase from **$197.56** on May 15, 2026, to **$235.71** on June 15, 2026.
   - The stock reached a peak of **$250.11** on June 5, 2026, indicating volatility.

4. **Volume**:
   - The trading volume has varied significantly, with a high of **19,108,336** shares traded on June 12, 2026, suggesting active trading and interest in the stock.

### Analysis

- **Recent Growth**: The stock has shown a strong upward trend over the past month, indicating positive market sentiment.
- **Volatility**: The stock price has fluctuated significantly, which could indicate a higher risk but also potential for higher returns.
- **Trading Volume**: High trading volumes can indicate strong investor interest, which is a positive sign.

### Conclusion

While the stock performance data shows a positive trend, it is essential to consider the company's financial health through its income statement and overall market conditions. Unfortunately, I couldn't retrieve the financials at this moment. 

For a more comprehensive analysis, I recommend checking the following:
- **Revenue Growth**: Look for consistent revenue growth in the income statement.
- **Profit Margins**: Assess the company's profitability through net income and operating margins.
- **Debt Levels**: Evaluate the company's debt-to-equity ratio to understand its financial leverage.
- **Market Conditions**: Consider the overall market conditions and industry trends affecting Cloudflare.

If you would like, I can attempt to retrieve the financial overview and income statement again. Would you like me to do that?